In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import os

RAW_DATA_DIR = os.environ.get('RAW_DATA_DIR', './')
FIGURES_DIR = os.environ.get('FIGURES_DIR', './')

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)


# --- PUBLICATION-QUALITY PLOT SETTINGS ---
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['legend.fontsize'] = 11
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.linewidth'] = 1.0 
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True       
mpl.rcParams['ytick.right'] = True
mpl.rcParams['figure.dpi'] = 150

# --- DATA LOADING HELPER ---
def load_data(filename, folder=RAW_DATA_DIR):
    path = os.path.join(folder, filename)
    if not os.path.exists(path):
        # Silent fail to avoid spamming output if a specific file is missing
        return None
    try:
        df = pd.read_csv(path, sep='\s+', comment='#', header=None, names=['idx', 'x', 'y'])
        return df
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None

# --- EXTENDED PALETTE ---
colors = {
    'exp':       'black',    # Truth
    'sim':       '#CC79A7',  # Simulation (Direct Problem) - Purple
    'geom':      '#E69F00',  # Geometric (Pre-calibration) - Orange/Yellow
    'eff':       '#D55E00',  # Effective (Calibrated) - Vermillion
    'topo':      '#0072B2',  # Topological - Blue
    'rand':      '#999999',  # Null Model - Grey
    'synthetic': '#009E73'   # Synthetic Gamma - Green
}

styles = {
    'exp':       'o',   # Points
    'sim':       '-',   # Solid
    'geom':      ':',   # Dotted
    'eff':       '--',  # Dashed
    'topo':      '-.',  # Dash-dot
    'rand':      '--',  # Dashed (often filled area)
    'synthetic': '-'    # Solid
}

print("Environment setup complete.")

## Testing my pore network library

In this section, we compare the PDF of the key physical quantities computed with my pore network code in several ways. Brief explanation of the legends:

* **Exp (Experimental):** Direct measurements from the porous medium (numerical sims).
* **Eff (Effective):** When talking about conductances: the real ones, that come from the actual ratio between real pressure drop and real flow rate. When talking about halfwidths: the ones required for a Brinkman tube to have the effective conductance.
* **Topo (Topological):** Results derived using real splitting fractions but solving the inverse problem for conductance.
* **Rand (Null Model):** Results derived assuming random splitting fractions ($U(0,1)$), ignoring local correlations.
* **Synthetic:** Results derived by sampling halfwidths from a Gamma distribution with the same mean and variance as the effective ones.

In [ ]:
def plot_comparison(metric_name, file_suffix, models, x_label_text, x_unit_inverse, x_lims=None, y_lims=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    
    # 1. Plot Rand/Null Model first (background fill)
    if 'rand' in models:
        filename = f"hist_{file_suffix}_rand.dat"
        df = load_data(filename)
        if df is not None:
            # Filled area for Rand
            #ax.fill_between(df['x'], df['y'], color=colors['rand'], alpha=0.15, label='_nolegend_')
            # Dashed line for Rand
            ax.plot(df['x'], df['y'], linestyle='--', color=colors['rand'], linewidth=1.5, label='Rand')
    
    # 2. Plot other models
    for model in models:
        if model == 'rand': continue # Already plotted
        
        filename = f"hist_{file_suffix}_{model}.dat"
        # Handle filename difference for synthetic
        if model == 'synthetic': filename = f"hist_{file_suffix}_synthetic.dat"

        df = load_data(filename)
        
        if df is not None:
            # --- LABELS (Sentence case) ---
            label_map = {
                'exp': 'Experimental',
                'sim': 'Simulation',
                'geom': 'Geometric',
                'eff': 'Effective',
                'topo': 'Topological',
                'synthetic': 'Synthetic'
            }
            label = label_map.get(model, model.capitalize())
            
            # --- STYLES ---
            if model == 'exp':
                # Continuous line + Points for Experimental
                ax.plot(df['x'], df['y'], marker='o', linestyle='-', color='black', 
                        markersize=5, mfc='white', markeredgewidth=1.0, linewidth=1.5, 
                        label=label, zorder=10)
            else:
                # Dashed lines for models
                # We vary the dash style slightly or keep uniform dashed
                ls = '--' 
                lw = 2.0
                ax.plot(df['x'], df['y'], linestyle=ls, color=colors[model], 
                        linewidth=lw, label=label)

    # --- AXIS LABELS ---
    # Sentence case for titles
    ax.set_xlabel(x_label_text)
    
    # Y Label: "Probability density (unit^-1)"
    if x_unit_inverse:
        ax.set_ylabel(f'Probability density ({x_unit_inverse})')
    else:
        ax.set_ylabel('Probability density')

    # --- SCALES & LIMITS ---
    # Linear scale everywhere (no set_yscale('log'))
    
    if x_lims: ax.set_xlim(x_lims)
    if y_lims: ax.set_ylim(y_lims)
    ax.set_ylim(bottom=0) # Density cannot be negative
    
    # Scientific notation for X axis
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
    
    # Legend: Sentence case is handled in label_map
    ax.legend(frameon=False, fontsize=11)
    
    # Grid
    ax.grid(True, linestyle=':', alpha=0.4)
    
    plt.tight_layout()
    plt.show()


# --- PLOT GENERATION ---

# 1. HALFWIDTHS (Geometry)
# Limit x to 0 - 1e-3 as requested
plot_comparison(
    metric_name='Halfwidths', 
    file_suffix='halfwidth', 
    models=['exp', 'eff', 'topo', 'rand', 'synthetic'], 
    x_label_text=r'Half-width $a$ [m]',
    x_unit_inverse=r'm$^{-1}$',
    #x_lims=(0, 1e-3)
)

# 2. CONDUCTANCES (Physics)
# Limit x to 0 - 1e-10 as requested
plot_comparison(
    metric_name='Conductances', 
    file_suffix='conductance', 
    models=['geom', 'eff', 'topo', 'rand', 'synthetic'], 
    x_label_text=r'Conductance $g$ [m$^3$/Pa$\cdot$s]',
    x_unit_inverse=r'Pa$\cdot$s/m$^3$',
    #x_lims=(0, 1e-10)
)

# 3. PRESSURE DROPS (Potentials)
plot_comparison(
    metric_name='Pressure Drops', 
    file_suffix='pressure_drop', 
    models=['exp', 'sim', 'topo', 'rand', 'synthetic'], 
    x_label_text=r'Pressure drop $|\Delta P|$ [Pa]',
    x_unit_inverse=r'Pa$^{-1}$'
)

# 4. FLOW RATES (Fluxes)
plot_comparison(
    metric_name='Flow Rates', 
    file_suffix='flow_rate', 
    models=['exp', 'sim', 'topo', 'rand', 'synthetic'], 
    x_label_text=r'Flow rate $Q$ [m$^3$/s]',
    x_unit_inverse=r's/m$^3$',
    #x_lims=None,
    #y_lims=(0,1.25e11)
)

# 5. SPLITTING FRACTIONS (Topology)
# Dimensionless, so no unit in Y label
plot_comparison(
    metric_name='Splitting Fractions', 
    file_suffix='splitting_fractions', 
    models=['exp', 'sim', 'rand', 'synthetic'], 
    x_label_text=r'Splitting fraction $\Omega$',
    x_unit_inverse=None 
)

## Effect of disorder in flow statistics

In this section, we analyze how the distributions of flow rates and pore masses evolve as we increase the geometric disorder (variance of the halfwidths distribution).

The disorder is controlled by the shape parameter $k$ of the Gamma distribution used to sample the half-widths.
* Low $k$ (e.g., 0.5): High disorder, broad distribution.
* High $k$ (e.g., 10.0): Low disorder, narrow (peaked) distribution.

We visualize the PDFs of the resulting flow statistics for selected values of $k$.

In [ ]:
def plot_evolution_k(metric_prefix, label_x, unit_inverse, folder=RAW_DATA_DIR, k_values=[0.5, 1.0, 2.0, 5.0, 10.0], xlims=None, ylims=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    
    # --- FIX: Modern Matplotlib Colormaps ---
    cmap = mpl.colormaps['viridis']
    norm = mpl.colors.Normalize(vmin=min(k_values), vmax=max(k_values))
    
    files_found = 0
    for k in sorted(k_values):
        # Filename format: hist_base_name_k_X.XX.dat
        filename = f"{metric_prefix}_k_{k:.2f}.dat"
        df = load_data(filename, folder=folder)
        
        if df is not None and not df.empty:
            color = cmap(norm(k))
            ax.plot(df['x'], df['y'], linewidth=2.0, color=color, label=f'$k = {k}$')
            files_found += 1
    
    if files_found == 0:
        print(f"Warning: No files found for '{metric_prefix}'.")

    # --- LABELS & STYLE ---
    ax.set_xlabel(label_x)
    
    if unit_inverse:
        ax.set_ylabel(f'Probability density ({unit_inverse})')
    else:
        ax.set_ylabel('Probability density')
    
    # Linear scale as requested
    ax.set_ylim(bottom=0)
    
    # Scientific notation for X
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))

    if(xlims):
        ax.set_xlim(xlims)
    if(ylims):
        ax.set_ylim(ylims)

    ax.legend(title=r'Shape param $k$', frameon=False, fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.4)
    
    plt.tight_layout()
    plt.show()



# Tu selección estricta
k_selection = [2.0, 5.0, 10.0]

print("--- 1. INPUT GEOMETRY ---")

# A. Half-widths (Recuerda haber añadido el save en C++)
plot_evolution_k(
    metric_prefix='hist_halfwidth', 
    label_x=r'Half-width $a$ [m]', 
    unit_inverse=r'm$^{-1}$',
    k_values=k_selection
)

# B. Conductances
plot_evolution_k(
    metric_prefix='hist_conductance', 
    label_x=r'Conductance $g$ [m$^3$/Pa$\cdot$s]', 
    unit_inverse=r'Pa$\cdot$s/m$^3$',
    k_values=k_selection,
    #xlims=(0, 0.6e-10),
    #ylims=(0, 0.7e11)
)

print("\n--- 2. POTENTIALS ---")

# C. Pressure Drops
plot_evolution_k(
    metric_prefix='hist_pressure_drop', 
    label_x=r'Pressure drop $|\Delta P|$ [Pa]', 
    unit_inverse=r'Pa$^{-1}$',
    k_values=k_selection,
    #xlims=(0, 0.4e-2)
)

print("\n--- 3. OUTPUT TRANSPORT ---")

# D. Flow Rates (Link Fluxes)
# Con tus límites manuales
plot_evolution_k(
    metric_prefix='hist_flow', 
    label_x=r'Flow rate $Q$ [m$^3$/s]', 
    unit_inverse=r's/m$^3$',
    k_values=k_selection,
    #xlims=(0, 2.0e-11),
    #ylims=(0, 1.2e11)
)

# E. Pore Flows / Masses (Nodal Fluxes)
# Usando el nombre correcto 'hist_pore_flows'
plot_evolution_k(
    metric_prefix='hist_pore_flows', 
    label_x=r'Pore flow $Q_{node}$ [m$^3$/s]', 
    unit_inverse=r's/m$^3$',
    k_values=k_selection
)

## Scaling law: halfwidth vs. flow rate

Finally, we test the scaling hypothesis linking the geometric disorder to the flow heterogeneity.

We plot the shape parameter of the pore flow rate distribution ($k_{pore}$) against the shape parameter of the input half-width distribution ($k_{hf}$).
* Recall that $k$ is the inverse of the normalized variance ($k \approx 1/CV^2$).
* A linear relationship implies a direct transfer of variance.

We compare the numerical results with the theoretical prediction, given by:
$$k_{pore} \approx \frac{3}{5} k_{hf}$$

In [ ]:
def plot_scaling_law(folder=RAW_DATA_DIR):
    filename = 'variance_scaling_curve.dat'
    path = os.path.join(folder, filename)
    
    if not os.path.exists(path):
        print("Scaling curve file not found.")
        return

    # Data format: Col 1 = NormVar_Input (1/k_in), Col 2 = NormVar_Output (1/k_out)
    data = pd.read_csv(path, sep='\s+', comment='#', header=None, names=['cv2_in', 'cv2_out'])
    
    # Transform to k space: k = 1 / CV^2
    # Filter out zeros to avoid division errors
    data = data[data['cv2_in'] > 1e-6]
    
    k_in = data['cv2_in']   # k_halfwidths (Input)
    k_out = data['cv2_out'] # k_pore_masses (Output)
    
    # --- Linear Fit ---
    def linear_model(x, m, c):
        return m * x + c
    
    popt, pcov = curve_fit(linear_model, k_in, k_out)
    slope = popt[0]
    intercept = popt[1]
    
    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # 1. Numerical Data (Points)
    # Using 'o' marker but following the requested color scheme (Simulation-like)
    ax.plot(k_in, k_out, marker='o', linestyle='', color='#003366', markersize=7, 
            markeredgecolor='white', markeredgewidth=1.0, label="Pore-network computations")
    
    # 2. Theoretical Line (y = 5/3 x)
    x_max = max(k_in)
    x_range = np.linspace(0, x_max * 1.05, 100)
    y_theory = (5.0/3.0) * x_range
    
    # Dashed line for theory (as it's a model/reference)
    ax.plot(x_range, y_theory, linestyle='--', color='#8B0000', linewidth=2.0, 
            label=r'$y = \frac{5}{3} x$')
    
    # Optional: Plot the actual fit if needed (dotted)
    # y_fit = linear_model(x_range, slope, intercept)
    # ax.plot(x_range, y_fit, linestyle=':', color='gray', label=f'Fit ($m={slope:.2f}$)')

    # --- LABELS & STYLE ---
    ax.set_xlabel(r'$\operatorname{Var}(A)/ \;\langle A \rangle^2 \;$')
    ax.set_ylabel(r'$\operatorname{Var}(Q_p)/ \;\langle Q_p \rangle^2 \;$')
    
    
    #ax.set_xscale('log')
    #ax.set_yscale('log')
    
    ax.legend(frameon=False, loc='upper left', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.4)
    
    plt.tight_layout()

    fig_path = os.path.join(FIGURES_DIR, 'variance_scaling_law.pdf')
    plt.savefig(fig_path, format='pdf', dpi=300, bbox_inches='tight')

    plt.show()
    
    print(f"Linear Fit Results:")
    print(f"  Slope     = {slope:.4f} (Theory: 0.6)")
    print(f"  Intercept = {intercept:.4f}")

plot_scaling_law()